In [1]:
#imports
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy.pool import NullPool
import psycopg2
from dotenv import load_dotenv
import os

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error

In [2]:
# test connection
# Load environment variables from .env
load_dotenv()

# Connect to the database
try:
    connection = psycopg2.connect(
    user=os.getenv("USER"),
    password=os.getenv("PASSWORD"),
    host=os.getenv("HOST"),
    port=os.getenv("PORT"),
    dbname=os.getenv("DBNAME"),
)
    print("Connection successful!")
    
    # Create a cursor to execute SQL queries
    cursor = connection.cursor()
    
    # Example query
    cursor.execute("SELECT NOW();")
    result = cursor.fetchone()
    print("Current Time:", result)

    # Close the cursor and connection
    cursor.close()
    connection.close()
    print("Connection closed.")

except Exception as e:
    print(f"Failed to connect: {e}")

Connection successful!
Current Time: (datetime.datetime(2026, 2, 26, 11, 26, 49, 615545, tzinfo=datetime.timezone.utc),)
Connection closed.


In [14]:
# download table from Supabase
load_dotenv()
conn = psycopg2.connect(
    user=os.getenv("USER"),
    password=os.getenv("PASSWORD"),
    host=os.getenv("HOST"),
    port=os.getenv("PORT"),
    dbname=os.getenv("DBNAME")
)

bmw_table = pd.read_sql("SELECT * FROM staging.bmw_data", conn)
hyundai_table = pd.read_sql("SELECT * FROM staging.hyundai_data", conn)
lada_table = pd.read_sql("SELECT * FROM staging.lada_data", conn)
conn.close()

e:\users\agabekov_ai\AppData\Local\Temp\ipykernel_25240\95099044.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bmw_table = pd.read_sql("SELECT * FROM staging.bmw_data", conn)
e:\users\agabekov_ai\AppData\Local\Temp\ipykernel_25240\95099044.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  hyundai_table = pd.read_sql("SELECT * FROM staging.hyundai_data", conn)
e:\users\agabekov_ai\AppData\Local\Temp\ipykernel_25240\95099044.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  lada_table = pd.read_sql("SELECT * FROM

In [3]:
# loading from local source
bmw_table = pd.read_csv(...)
hyundai_table = pd.read_csv(...)
lada_table = pd.read_csv(...)

In [4]:
# formatting
bmw_table.drop(columns = ['country', 'region', 'channel'], inplace = True)
hyundai_table.drop(columns = 'country', inplace = True)
lada_table.drop(columns = 'country', inplace = True)
tables = [bmw_table, hyundai_table, lada_table]
brands = ['BMW', 'Hyundai', 'Lada']
for i, (table, brand) in enumerate(zip(tables, brands)):
    table.insert(2, 'brand', f'{brand}')
   

In [5]:
# concat brand tables for model
total = pd.concat(
    [bmw_table, hyundai_table, lada_table],
    axis = 0,
    join = 'outer',
    ignore_index = True).sort_values(by = ['year', 'month'])

total.insert(0, 'date', pd.to_datetime(total[['year', 'month']].assign(day = 1)))
year_month = total[['year', 'month']]
total.drop(columns = ['year', 'month'], inplace = True)
total = pd.concat([total, year_month], axis = 1)
total['quarter'] = total['date'].dt.quarter
total[['year', 'month', 'revenue', 'month_qnt']] = total[['year', 'month', 'revenue', 'month_qnt']].astype('int64')
total.insert(5, 'avg_price', total.revenue / total.month_qnt)
total.head()

,date,brand,model,revenue,month_qnt,avg_price,year,month,quarter
1,2021-01-01,BMW,BMW 3 Series,107774,5,21554.8,2021,1,1
3,2021-01-01,BMW,BMW Z4,78848,2,39424.0,2021,1,1
20,2021-01-01,BMW,BMW X1,40722,2,20361.0,2021,1,1
29,2021-01-01,BMW,BMW 8 Series,115478,4,28869.5,2021,1,1
71,2021-01-01,BMW,BMW X2,102582,1,102582.0,2021,1,1


In [6]:
# calc lag and MA per Brand-Model
total['lag_1_qnt'] = total.groupby(['brand', 'model'])['month_qnt'].shift(1)
total['lag_3_qnt'] = total.groupby(['brand', 'model'])['month_qnt'].shift(3)
total['lag_6_qnt'] = total.groupby(['brand', 'model'])['month_qnt'].shift(6)

total['lag_1_price'] = total.groupby(['brand', 'model'])['avg_price'].shift(1)
total['lag_3_price'] = total.groupby(['brand', 'model'])['avg_price'].shift(3)
total['lag_6_price'] = total.groupby(['brand', 'model'])['avg_price'].shift(6)

total['MA_3_qnt'] = total.groupby(['brand', 'model'])['month_qnt'].shift(1).rolling(3).mean()
total['MA_6_qnt'] = total.groupby(['brand', 'model'])['month_qnt'].shift(1).rolling(6).mean()

total['MA_3_price'] = total.groupby(['brand', 'model'])['avg_price'].shift(1).rolling(3).mean()
total['MA_6_price'] = total.groupby(['brand', 'model'])['avg_price'].shift(1).rolling(6).mean()

# delete first_values
total = total[~total['date'].between('2021-01-01','2021-06-01')]

# remove null
total.fillna(method = 'ffill', inplace = True)

e:\users\agabekov_ai\AppData\Local\Temp\ipykernel_17944\1188077041.py:20: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  total.fillna(method = 'ffill', inplace = True)


In [7]:
total.head()

,date,brand,model,revenue,month_qnt,avg_price,year,month,quarter,lag_1_qnt,lag_3_qnt,lag_6_qnt,lag_1_price,lag_3_price,lag_6_price,MA_3_qnt,MA_6_qnt,MA_3_price,MA_6_price
19,2021-07-01,BMW,BMW 1 Series,110266,1,110266.0,2021,7,3,8.0,2.0,2.0,15759.500000,24687.00,17601.00,7.000000,6.166667,9198.477778,6999.738889
32,2021-07-01,BMW,BMW M8,51943,2,25971.5,2021,7,3,4.0,1.0,5.0,26791.750000,49887.00,23201.00,5.000000,6.166667,16388.194444,10770.613889
52,2021-07-01,BMW,BMW 6 Series,79179,5,15835.8,2021,7,3,1.0,2.0,4.0,112901.000000,41317.50,10344.25,4.333333,5.000000,51817.416667,28675.238889
58,2021-07-01,BMW,BMW M5,30036,3,10012.0,2021,7,3,3.0,4.0,2.0,29900.333333,9829.75,24857.50,2.666667,4.833333,56531.027778,32864.752778
62,2021-07-01,BMW,BMW i8,53039,1,53039.0,2021,7,3,3.0,2.0,5.0,10571.000000,20428.00,13497.40,2.333333,3.666667,51124.111111,33756.152778


In [8]:
# drop dates 
total.drop(columns = 'date', inplace = True)

In [9]:
# insert time trend to improve model accuracy 
total.insert(7, 'time_trend', (total['year'] - total['year'].min()) * 12 + total['month'])

In [10]:
total.head()

,brand,model,revenue,month_qnt,avg_price,year,month,time_trend,quarter,lag_1_qnt,lag_3_qnt,lag_6_qnt,lag_1_price,lag_3_price,lag_6_price,MA_3_qnt,MA_6_qnt,MA_3_price,MA_6_price
19,BMW,BMW 1 Series,110266,1,110266.0,2021,7,7,3,8.0,2.0,2.0,15759.500000,24687.00,17601.00,7.000000,6.166667,9198.477778,6999.738889
32,BMW,BMW M8,51943,2,25971.5,2021,7,7,3,4.0,1.0,5.0,26791.750000,49887.00,23201.00,5.000000,6.166667,16388.194444,10770.613889
52,BMW,BMW 6 Series,79179,5,15835.8,2021,7,7,3,1.0,2.0,4.0,112901.000000,41317.50,10344.25,4.333333,5.000000,51817.416667,28675.238889
58,BMW,BMW M5,30036,3,10012.0,2021,7,7,3,3.0,4.0,2.0,29900.333333,9829.75,24857.50,2.666667,4.833333,56531.027778,32864.752778
62,BMW,BMW i8,53039,1,53039.0,2021,7,7,3,3.0,2.0,5.0,10571.000000,20428.00,13497.40,2.333333,3.666667,51124.111111,33756.152778


In [15]:
# def functions 

# func "prepare data" separates datasets and adds dummy variables
def prepare_data(df, dummy = 0, value = 'price'):
    df_list = []
    if dummy == 1:
        df = pd.get_dummies(df, columns = ['brand', 'model'], drop_first = True)

    train = df.query('year == 2021')
    valid = df.query('year == 2022')
    if value == 'qnt':
        for df in (train, valid):
            df_list.append(df.drop(columns = ['revenue', 'month_qnt', 'avg_price']))
            df_list.append(np.log1p(df['month_qnt']))
    else:
        for df in (train, valid):
            df_list.append(df.drop(columns = ['revenue', 'month_qnt', 'avg_price']))
            df_list.append(np.log1p(df['avg_price']))            
    return df_list

# func "model_test" creates models and tests them against 3 error metrics, returning a dataframe with the results
def model_test():
    valid_errors_df = pd.DataFrame()
    ln_model_q = LinearRegression()
    RF_model_q = RandomForestRegressor(n_estimators=500, random_state=42)
    GB_model_q = CatBoostRegressor(iterations=1000, depth=8, learning_rate=0.05, random_state=42)
    ln_model_p = LinearRegression()
    RF_model_p = RandomForestRegressor(n_estimators=500, random_state=42)
    GB_model_p = CatBoostRegressor(iterations=1000, depth=8, learning_rate=0.05, random_state=42)
    models = {'ln_model_q' : (ln_model_q, 1, 'qnt'), 'RF_model_q' : (RF_model_q, 1, 'qnt'), 'GB_model_q' : (GB_model_q, 0, 'qnt'), 'ln_model_p' : (ln_model_p, 1, 'price'), 'RF_model_p' : (RF_model_p, 1, 'price'), 'GB_model_p' : (GB_model_p, 0, 'price')}
    metrics = [mean_absolute_error, mean_squared_error, mean_absolute_percentage_error]
    metric_names = ['MAE', 'RMSE', 'MAPE']

    for model_name, (model, dummy, value) in models.items():
        X_train, y_train, X_valid, y_valid = prepare_data(total, dummy, value)
        if dummy == 1:
            model.fit(X_train, y_train)    
        else: 
            model.fit(X_train, y_train, cat_features = ['brand', 'model'], verbose = 100)
        y_pred = np.expm1(model.predict(X_valid))    
        temp = pd.DataFrame({'model' : [f'{model_name}']})
        for metric_name, metric in zip(metric_names, metrics):
            if metric_name == 'RMSE':
                temp[metric_name] = np.sqrt(metric(np.expm1(y_valid), y_pred))
            else:
                temp[metric_name] = metric(np.expm1(y_valid), y_pred)
        valid_errors_df = pd.concat([valid_errors_df, temp], ignore_index = True)
    return valid_errors_df

In [16]:
model_test()

0:	learn: 0.4783147	total: 210ms	remaining: 3m 30s
100:	learn: 0.3192106	total: 8.45s	remaining: 1m 15s
200:	learn: 0.2671488	total: 17.3s	remaining: 1m 8s
300:	learn: 0.2240137	total: 25.3s	remaining: 58.8s
400:	learn: 0.1906197	total: 34s	remaining: 50.8s
500:	learn: 0.1643002	total: 42.2s	remaining: 42.1s
600:	learn: 0.1439130	total: 51.2s	remaining: 34s
700:	learn: 0.1243408	total: 1m	remaining: 25.7s
800:	learn: 0.1108923	total: 1m 9s	remaining: 17.2s
900:	learn: 0.0965938	total: 1m 18s	remaining: 8.58s
999:	learn: 0.0854207	total: 1m 26s	remaining: 0us
0:	learn: 0.7930327	total: 26.6ms	remaining: 26.6s
100:	learn: 0.4500994	total: 7.53s	remaining: 1m 7s
200:	learn: 0.3383119	total: 14.1s	remaining: 56s
300:	learn: 0.2656872	total: 20.1s	remaining: 46.7s
400:	learn: 0.2053167	total: 26.1s	remaining: 38.9s
500:	learn: 0.1607752	total: 32.1s	remaining: 31.9s
600:	learn: 0.1299837	total: 38s	remaining: 25.2s
700:	learn: 0.1071253	total: 45.6s	remaining: 19.5s
800:	learn: 0.0907247	to

,model,MAE,RMSE,MAPE
0,ln_model_q,1.566776,2.152213,0.482346
1,RF_model_q,1.596952,2.133035,0.493801
2,GB_model_q,1.558172,2.112443,0.486341
3,ln_model_p,11276.424331,20791.599088,0.407613
4,RF_model_p,11000.550975,20490.844900,0.412322
5,GB_model_p,11547.195791,20857.597999,0.433440


In [23]:
# according to the MAPE criterion, the linear model showed the best results
# check ElasticNet as an opportunity to reduce MAPE. Add GridSearch to find the best parameters.
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV

model = ElasticNet()

total_d = pd.get_dummies(total, columns = ['brand', 'model'], drop_first = True)

final_errors_df = pd.DataFrame()


param_grid = {'alpha' : [1, 0.1, 0.001], 'l1_ratio' : [0.1, 0.3, 0.6, 0.9], 'max_iter' : [1000, 5000, 10000]}
best_model_q = GridSearchCV(estimator = model, param_grid = param_grid, cv = 10, scoring = 'neg_mean_absolute_percentage_error')
best_model_p = GridSearchCV(estimator = model, param_grid = param_grid, cv = 10, scoring = 'neg_mean_absolute_error')

X_train = total_d.query('year in [2021,2022]').drop(columns = ['revenue', 'month_qnt', 'avg_price'])
y_train_q = total_d.query('year in [2021,2022]')['month_qnt']
y_train_p = np.log1p(total_d.query('year in [2021,2022]')['avg_price'])

X_test = total_d.query('year == 2023').drop(columns = ['revenue', 'month_qnt', 'avg_price'])
y_test_q = total_d.query('year == 2023')['month_qnt']
y_test_p = np.log1p(total_d.query('year == 2023')['avg_price'])

best_model_q.fit(X_train, y_train_q)
best_model_p.fit(X_train, y_train_p)

y_pred_q = best_model_q.predict(X_test)
y_pred_p = np.expm1(best_model_p.predict(X_test))

metrics = [mean_absolute_error, mean_squared_error, mean_absolute_percentage_error]
metric_names = ['MAE', 'RMSE', 'MAPE']

for metric_name, metric in zip(metric_names, metrics):
    if metric_name == 'RMSE':
        final_errors_df[f'{metric_name}_q'] = [np.sqrt(metric(y_test_q, y_pred_q))]
        final_errors_df[f'{metric_name}_p'] = [np.sqrt(metric(np.expm1(y_test_p), y_pred_p))]
    else:
        final_errors_df[f'{metric_name}_q'] = [metric(y_test_q, y_pred_q)]
        final_errors_df[f'{metric_name}_p'] = [metric(np.expm1(y_test_p), y_pred_p)]


final_errors_df.head(20)

,MAE_q,MAE_p,RMSE_q,RMSE_p,MAPE_q,MAPE_p
0,1.523203,11030.328503,2.035626,20845.196358,0.485391,0.371087


In [44]:
# according to the MAPE criterion, the linear model showed the best results
# check ElasticNet as an opportunity to reduce MAPE. Add GridSearch to find the best parameters
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV

model = ElasticNet()

total_d = pd.get_dummies(total, columns = ['brand', 'model'], drop_first = True)

final_errors_df = pd.DataFrame(index = ['test', 'baseline', 'accuracy'])


param_grid = {'alpha' : [1, 0.1, 0.001], 'l1_ratio' : [0.1, 0.3, 0.6, 0.9], 'max_iter' : [1000, 5000, 10000]}
best_model_q = GridSearchCV(estimator = model, param_grid = param_grid, cv = 10, scoring = 'neg_mean_absolute_percentage_error')
best_model_p = GridSearchCV(estimator = model, param_grid = param_grid, cv = 10, scoring = 'neg_mean_absolute_error')

X_train = total_d.query('year in [2021,2022]').drop(columns = ['revenue', 'month_qnt', 'avg_price'])
y_train_q = total_d.query('year in [2021,2022]')['month_qnt']
y_train_p = np.log1p(total_d.query('year in [2021,2022]')['avg_price'])

X_test = total_d.query('year == 2023').drop(columns = ['revenue', 'month_qnt', 'avg_price'])
y_test_q = total_d.query('year == 2023')['month_qnt']
y_test_p = np.log1p(total_d.query('year == 2023')['avg_price'])

best_model_q.fit(X_train, y_train_q)
best_model_p.fit(X_train, y_train_p)

y_pred_q = best_model_q.predict(X_test)
y_pred_p = np.expm1(best_model_p.predict(X_test))

# add baseline for further evaluation of the model

y_baseline_pred_q = np.repeat(y_train_q.mean(), len(y_test_q))
y_baseline_pred_p = np.repeat(np.expm1(y_train_p).mean(), len(y_test_p))


# check ElasticNet model using 3 metrics and add the baseline to the final dataframe

metrics = [mean_absolute_error, mean_squared_error, mean_absolute_percentage_error]
metric_names = ['MAE', 'RMSE', 'MAPE']

for metric_name, metric in zip(metric_names, metrics):
    if metric_name == 'RMSE':
        final_errors_df.loc['test', f'{metric_name}_q'] = np.sqrt(metric(y_test_q, y_pred_q))
        final_errors_df.loc['test', f'{metric_name}_p'] = np.sqrt(metric(np.expm1(y_test_p), y_pred_p))
        final_errors_df.loc['baseline', f'{metric_name}_q'] = np.sqrt(metric(y_test_q, y_baseline_pred_q))
        final_errors_df.loc['baseline', f'{metric_name}_p'] = np.sqrt(metric(np.expm1(y_test_p), y_baseline_pred_p))
    else:
        final_errors_df.loc['test', f'{metric_name}_q'] = metric(y_test_q, y_pred_q)
        final_errors_df.loc['test', f'{metric_name}_p'] = metric(np.expm1(y_test_p), y_pred_p)
        final_errors_df.loc['baseline', f'{metric_name}_q'] = metric(y_test_q, y_baseline_pred_q)
        final_errors_df.loc['baseline', f'{metric_name}_p'] = metric(np.expm1(y_test_p), y_baseline_pred_p)

#final_errors_df.index = ['Test']
final_errors_df.head(20)

,MAE_q,MAE_p,RMSE_q,RMSE_p,MAPE_q,MAPE_p
test,1.523203,11030.328503,2.035626,20845.196358,0.485391,0.371087
baseline,1.836963,17490.422245,2.639738,23923.021688,0.698100,1.131355
accuracy,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
# add a comparison of the model result with the baseline prediction based on the average

for column in final_errors_df.columns:
    final_errors_df.loc['accuracy', column] = final_errors_df.loc['baseline', column] - final_errors_df.loc['test', column]

In [47]:
final_errors_df.head()

,MAE_q,MAE_p,RMSE_q,RMSE_p,MAPE_q,MAPE_p
test,1.523203,11030.328503,2.035626,20845.196358,0.485391,0.371087
baseline,1.836963,17490.422245,2.639738,23923.021688,0.698100,1.131355
accuracy,0.313760,6460.093742,0.604112,3077.825330,0.212709,0.760268
